# Figure S2

This notebook evaluates squared-fitness ruggedness inference under A. thaliana nucleotide mutation models. Panels A–C use synthetic NK landscapes to compare local decay with $\rho_{NK}$, and Panels D–F measure start-sampling accuracy on full-nucleotide empirical landscapes.

## Setup

Load the NK and empirical landscapes, mutation-only simulation and fitting functions, codon map, analytical product-kernel calculations, serialisation tools, and plotting utilities. Define the shared parameters, file paths, and overwrite or plot-only controls.

Cached raw and processed products can be installed from the repository root with `python slide/download_zenodo_data.py`. The notebook can generate missing products when its overwrite and plot-only settings permit it.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pickle
import subprocess
import sys
from pathlib import Path

import jax.random as jr
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from tqdm.auto import tqdm

from slide.data_generation import (
    nk_grid_pairs,
    ordered_unique_pairs,
    random_start,
    run_nk_start_averaged_diffusion,
)
from slide.direvo_functions import (
    CODON_MAPPER,
    get_single_decay_rate,
    get_single_decay_rate_IK_v2,
)
from slide.utils import (
    FIGURE_LABEL_SIZE,
    FIGURE_LEGEND_SIZE,
    FIGURE_TICK_SIZE,
    FIGURE_TITLE_SIZE,
    PANEL_LETTER_SIZE,
    get_figures_dir,
    get_processed_data_dir,
    get_raw_data_dir,
    load_pickle,
    save_pickle,
)
from slide_config import get_slide_data_dir

OVERWRITE_RAW_PKL: bool = False
OVERWRITE_PROCESSED_PKL: bool = False
PLOT_ONLY: bool = False
SAVE_FIGURES: bool = True
PANEL_DPI: int = 350
SAVE_TYPES: tuple[str, ...] = ("pdf", "png", "eps")

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
SLIDE_DATA_DIR = Path(get_slide_data_dir())
REPO_ROOT = Path.cwd()

MODEL_KEYS: tuple[str, ...] = (
    "nuc_uniform",
    "nuc_a_thaliana_weighted",
    "nuc_a_thaliana_directed",
)
MODEL_TITLES: dict[str, str] = {
    "nuc_uniform": "Uniform mutation",
    "nuc_a_thaliana_weighted": r"Weighted $\mathit{A.\ thaliana}$",
    "nuc_a_thaliana_directed": r"Directed $\mathit{A.\ thaliana}$",
}
LANDSCAPE_KEYS: tuple[str, ...] = ("gb1", "tev", "trpb", "pard3")
LANDSCAPE_NAMES: tuple[str, ...] = ("GB1", "TEV", "TrpB", "ParD3")
LANDSCAPE_COLORS: tuple[str, ...] = ("tab:orange", "tab:green", "tab:blue", "tab:red")
LANDSCAPE_MARKERS: tuple[str, ...] = ("o", "^", "s", "D")

ABC_PROCESSED_PATH = PROCESSED_DATA_DIR / "figureS2_a_thaliana_nk_local_gmu_processed.pkl"
DF_PER_LANDSCAPE_PROCESSED_PATHS: dict[str, Path] = {
    landscape: PROCESSED_DATA_DIR / f"figure6_full_nuc_gmu_a_thaliana_{landscape}_sampling_75steps_processed.pkl"
    for landscape in LANDSCAPE_KEYS
}
ANALYTICS_PER_LANDSCAPE_PATHS: dict[str, Path] = {
    landscape: PROCESSED_DATA_DIR / f"figure6_full_nuc_gmu_{landscape}_kernel_analytics_processed.pkl"
    for landscape in LANDSCAPE_KEYS
}
LANDSCAPE_FILE_BY_NAME: dict[str, str] = {
    "GB1": "GB1_landscape_array.pkl",
    "TEV": "TEV_landscape_array.pkl",
    "TrpB": "TrpB_landscape_array.pkl",
    "ParD3": "E3_landscape_array.pkl",
}
ABC_RAW_PATHS: dict[str, Path] = {
    model: RAW_DATA_DIR / f"figure6_ecoli_nk_{model}_raw.pkl"
    for model in MODEL_KEYS
}

ABC_N_VALUES: tuple[int, ...] = (10, 14, 18, 23, 27, 32, 36, 41, 45, 50)
ABC_NUM_ALLELES: int = 4
ABC_NUM_K_VALUES_PER_N: int = 10
ABC_NUM_LANDSCAPES: int = 25
ABC_NUM_STARTS: int = 25
ABC_NUM_REPLICATES: int = 5
ABC_POPULATION_SIZE: int = 2_500
ABC_NUM_GENERATIONS: int = 25
ABC_TOTAL_MUTATION_RATE: float = 0.5

DF_NUM_GENERATIONS: int = 75
DF_TOTAL_MUTATION_RATE: float = 0.1
RANDOM_SEED: int = 42

print(f"PLOT_ONLY={PLOT_ONLY}, OVERWRITE_RAW_PKL={OVERWRITE_RAW_PKL}, "
      f"OVERWRITE_PROCESSED_PKL={OVERWRITE_PROCESSED_PKL}")


def save_figure(fig: Figure, stem: str, *, bbox_inches: str = "tight") -> None:
    """Save one figure in every configured format.

    Parameters:
    - fig: Figure
        Figure to save.
    - stem: str
        Output filename stem.
    - bbox_inches: str
        Matplotlib bounding-box mode.

    Returns:
    - None
        Files are written below ``FIGURES_DIR``.
    """
    for suffix in SAVE_TYPES:
        destination = FIGURES_DIR / suffix
        destination.mkdir(parents=True, exist_ok=True)
        fig.savefig(destination / f"{stem}.{suffix}", dpi=PANEL_DPI, bbox_inches=bbox_inches)


def add_panel_letter(ax: Axes, letter: str) -> None:
    """Add a manuscript panel letter.

    Parameters:
    - ax: Axes
        Axis receiving the label.
    - letter: str
        Panel letter.

    Returns:
    - None
        The axis is modified in place.
    """
    ax.text(-0.14, 1.10, letter, transform=ax.transAxes,
            fontsize=PANEL_LETTER_SIZE, fontweight="bold", va="top", ha="left")


## Mutation Kernels

Construct the three single-site nucleotide operators used in Figure S2: uniform mutation, a symmetric A. thaliana-weighted kernel obtained by Sinkhorn scaling the transpose-averaged directed kernel, and the original directed row-stochastic A. thaliana kernel. Symmetric models use a uniform stationary distribution; the directed model uses its calculated stationary distribution.

In [ ]:
def symmetric_sinkhorn_kernel(kernel: np.ndarray, tolerance: float = 1e-13) -> np.ndarray:
    """Create a symmetric doubly-stochastic kernel by diagonal scaling.

    Parameters:
    - kernel: np.ndarray
        Non-negative square base kernel.
    - tolerance: float
        Maximum permitted row-sum error.

    Returns:
    - np.ndarray
        Symmetric, doubly-stochastic kernel with the input zero pattern.
    """
    symmetric = 0.5 * (np.asarray(kernel, dtype=float) + np.asarray(kernel, dtype=float).T)
    scale = np.ones(symmetric.shape[0], dtype=float)
    for _ in range(100_000):
        row_sums = scale * (symmetric @ scale)
        if np.max(np.abs(row_sums - 1.0)) < tolerance:
            break
        scale *= np.sqrt(1.0 / row_sums)
    else:
        raise RuntimeError("Symmetric Sinkhorn scaling did not converge.")
    return scale[:, None] * symmetric * scale[None, :]


def stationary_distribution(kernel: np.ndarray) -> np.ndarray:
    """Return the normalized stationary distribution of a row-stochastic kernel.

    Parameters:
    - kernel: np.ndarray
        Irreducible row-stochastic transition kernel.

    Returns:
    - np.ndarray
        Positive stationary probability vector.
    """
    eigenvalues, eigenvectors = np.linalg.eig(np.asarray(kernel, dtype=float).T)
    index = int(np.argmin(np.abs(eigenvalues - 1.0)))
    stationary = np.real(eigenvectors[:, index])
    if stationary.sum() < 0:
        stationary *= -1
    stationary /= stationary.sum()
    return stationary


uniform_kernel = (np.ones((4, 4)) - np.eye(4)) / 3.0
a_thaliana_directed_kernel = np.asarray(
    np.load(REPO_ROOT / "other_data" / "normed_a_thaliana_matrix.npy"), dtype=float
)
a_thaliana_weighted_kernel = symmetric_sinkhorn_kernel(a_thaliana_directed_kernel)
MUTATION_KERNELS: dict[str, np.ndarray] = {
    "nuc_uniform": uniform_kernel,
    "nuc_a_thaliana_weighted": a_thaliana_weighted_kernel,
    "nuc_a_thaliana_directed": a_thaliana_directed_kernel,
}

for model, kernel in MUTATION_KERNELS.items():
    if kernel.shape != (4, 4) or np.any(kernel < 0):
        raise ValueError(f"Invalid mutation kernel for {model}.")
    if not np.allclose(kernel.sum(axis=1), 1.0, atol=1e-12):
        raise ValueError(f"Mutation kernel {model} is not row-stochastic.")
if not np.allclose(a_thaliana_weighted_kernel, a_thaliana_weighted_kernel.T, atol=1e-12):
    raise ValueError("Weighted A. thaliana kernel is not symmetric.")
if not np.allclose(a_thaliana_weighted_kernel.sum(axis=0), 1.0, atol=1e-12):
    raise ValueError("Weighted A. thaliana kernel is not column-stochastic.")
if not np.allclose(np.diag(a_thaliana_weighted_kernel), 0.0, atol=1e-14):
    raise ValueError("Weighted A. thaliana kernel does not preserve the zero diagonal.")


## Raw Data — Synthetic NK Landscapes for Panels A–C

Sample 100 $(N,K)$ pairs over $10\leq N\leq50$ with nucleotide alphabet size $A=4$. For each mutation kernel, generate the configured number of NK landscapes, starting genotypes, and population replicates. Mutation acts at the encoded total sequence rate divided across sites. The raw trajectory tensor retains pair, landscape, start, replicate, and generation axes so local squared-fitness decay can be calculated after averaging population replicates.

In [ ]:
raw_pairs = nk_grid_pairs((10, 50), ABC_NUM_K_VALUES_PER_N, K_start=0)
abc_nk_pairs = ordered_unique_pairs(raw_pairs)
if len(abc_nk_pairs) != 100:
    raise AssertionError("Expected 100 Figure S2 A-C NK pairs.")

abc_raw_payloads: dict[str, dict[str, object]] = {}
if not PLOT_ONLY:
    pair_keys = jr.split(jr.PRNGKey(RANDOM_SEED), len(abc_nk_pairs))
    for model in MODEL_KEYS:
        path = ABC_RAW_PATHS[model]
        if path.exists() and not OVERWRITE_RAW_PKL:
            abc_raw_payloads[model] = load_pickle(path)
            continue
        trajectories = np.empty(
            (len(abc_nk_pairs), ABC_NUM_LANDSCAPES, ABC_NUM_STARTS,
             ABC_NUM_REPLICATES, ABC_NUM_GENERATIONS), dtype=np.float32,
        )
        starts_padded = np.full(
            (len(abc_nk_pairs), ABC_NUM_LANDSCAPES, ABC_NUM_STARTS, max(ABC_N_VALUES)),
            -1, dtype=np.int8,
        )
        landscape_keys_saved = np.empty(
            (len(abc_nk_pairs), ABC_NUM_LANDSCAPES, 2), dtype=np.uint32,
        )
        for pair_index, (pair_key, pair) in enumerate(zip(pair_keys, abc_nk_pairs, strict=True)):
            n_sites, k_value = pair
            landscape_keys = jr.split(pair_key, ABC_NUM_LANDSCAPES)
            for landscape_index, landscape_key in enumerate(landscape_keys):
                start_keys = jr.split(
                    jr.fold_in(landscape_key, 100_000 + landscape_index), ABC_NUM_STARTS
                )
                starts = np.asarray([
                    random_start(key, n_sites=n_sites, num_alleles=ABC_NUM_ALLELES)
                    for key in start_keys
                ])
                starts_padded[pair_index, landscape_index, :, :n_sites] = starts
                landscape_keys_saved[pair_index, landscape_index] = np.asarray(landscape_key)
                trajectories[pair_index, landscape_index] = run_nk_start_averaged_diffusion(
                    rng_key=landscape_key,
                    trajectory_rng_key=landscape_key,
                    n_sites=n_sites,
                    k=k_value,
                    num_alleles=ABC_NUM_ALLELES,
                    starts=starts,
                    popsize=ABC_POPULATION_SIZE,
                    mutation_rate_per_site=ABC_TOTAL_MUTATION_RATE / n_sites,
                    num_reps_per_start=ABC_NUM_REPLICATES,
                    num_steps=ABC_NUM_GENERATIONS,
                    mutation_matrix=MUTATION_KERNELS[model],
                    return_replicates=True,
                )
        payload = {
            "data": {
                "fitness_trajectories": trajectories,
                "start_coordinates_padded": starts_padded,
                "landscape_keys": landscape_keys_saved,
                "mutation_kernel": MUTATION_KERNELS[model],
            },
            "params": {
                "model": model, "A": ABC_NUM_ALLELES, "nk_pairs": abc_nk_pairs,
                "num_landscapes": ABC_NUM_LANDSCAPES, "num_starts": ABC_NUM_STARTS,
                "num_population_replicates": ABC_NUM_REPLICATES,
                "population_size": ABC_POPULATION_SIZE, "M": ABC_NUM_GENERATIONS,
                "total_mutation_rate": ABC_TOTAL_MUTATION_RATE, "seed": RANDOM_SEED,
            },
            "metadata": {"paper_reference": "Figure S2A-C"},
        }
        save_pickle(payload, path)
        abc_raw_payloads[model] = payload
else:
    print("PLOT_ONLY=True: skipping Figure S2 A-C raw generation.")


## Processing — Synthetic NK Local Decay for Panels A–C

Average population replicates at fixed starting genotype to estimate $F_\mu(\nu)$, square the result to obtain local $G_\mu(\nu)$, and fit one squared-decay rate per start. Group successful estimates into ten bins of $\rho_{NK}=(K+1)/N$ and calculate the mean, standard deviation, and count in each bin for the uniform, symmetric weighted, and directed kernels.

In [ ]:
def process_abc_payload(raw_by_model: dict[str, dict[str, object]]) -> dict[str, object]:
    """Fit and bin local squared-fitness decay rates for panels A-C.

    Parameters:
    - raw_by_model: dict[str, dict[str, object]]
        Replicate-level raw trajectory payloads.

    Returns:
    - dict[str, object]
        Per-start fits and binned summaries.
    """
    output: dict[str, object] = {}
    for model in MODEL_KEYS:
        raw_payload = raw_by_model[model]
        trajectories = np.asarray(raw_payload["data"]["fitness_trajectories"], dtype=float)
        pairs = tuple((int(n), int(k)) for n, k in raw_payload["params"]["nk_pairs"])
        f_mu = trajectories.mean(axis=3)
        g_mu = np.square(f_mu)
        rho_local = np.full(g_mu.shape[:-1], np.nan, dtype=float)
        fit_success = np.zeros(g_mu.shape[:-1], dtype=bool)
        for index in tqdm(
            np.ndindex(g_mu.shape[:-1]),
            total=int(np.prod(g_mu.shape[:-1])),
            desc=f"Fitting A-C {MODEL_TITLES[model]}",
            leave=False,
        ):
            try:
                rho_local[index] = float(get_single_decay_rate(
                    g_mu[index], mut=2.0 * ABC_TOTAL_MUTATION_RATE,
                    num_steps=ABC_NUM_GENERATIONS,
                )[0])
                fit_success[index] = True
            except (RuntimeError, ValueError, FloatingPointError):
                continue
        rho_nk = np.asarray([(k + 1) / n for n, k in pairs], dtype=float)
        repeated = np.broadcast_to(rho_nk[:, None, None], rho_local.shape)
        valid_rho = repeated[fit_success]
        valid_estimates = rho_local[fit_success]
        bin_edges = np.linspace(0.0, 1.0, 11)
        bin_indices = np.clip(np.digitize(valid_rho, bin_edges, right=True) - 1, 0, 9)
        binned_rho, binned_mean, binned_std, binned_counts = [], [], [], []
        for bin_index in range(10):
            selected = bin_indices == bin_index
            if np.any(selected):
                binned_rho.append(float(valid_rho[selected].mean()))
                binned_mean.append(float(valid_estimates[selected].mean()))
                binned_std.append(float(valid_estimates[selected].std()))
                binned_counts.append(int(selected.sum()))
        output[model] = {
            "rho_2_local": rho_local,
            "fit_success": fit_success,
            "rho_NK": rho_nk,
            "binned_rho_NK": np.asarray(binned_rho),
            "binned_mean": np.asarray(binned_mean),
            "binned_std": np.asarray(binned_std),
            "binned_counts": np.asarray(binned_counts),
        }
    return {
        "data": output,
        "params": {"models": MODEL_KEYS, "M": ABC_NUM_GENERATIONS,
                   "total_mutation_rate": ABC_TOTAL_MUTATION_RATE},
        "metadata": {"paper_reference": "Figure S2A-C"},
    }


if ABC_PROCESSED_PATH.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    print(f"Loading pre-processed Figure S2 A-C payload from {ABC_PROCESSED_PATH}")
    figure_s2_abc_payload = load_pickle(ABC_PROCESSED_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(f"PLOT_ONLY=True requires {ABC_PROCESSED_PATH}")
else:
    print(f"Processing Figure S2 A-C payload: {ABC_PROCESSED_PATH}")
    figure_s2_abc_payload = process_abc_payload(abc_raw_payloads)
    save_pickle(figure_s2_abc_payload, ABC_PROCESSED_PATH)


## Raw Data — Full-Nucleotide Empirical Landscapes for Panels D–F

Generate or load compact $G_\mu$ products for GB1, TrpB, TEV, and ParD3 expanded from amino-acid to nucleotide space with the codon map. Invalid and stop codons receive the minimum landscape fitness. For each A. thaliana mutation model, the products contain 20 start orderings, increasing start-prefix counts, and the configured mutation-only generations, enabling sampling accuracy to be assessed without storing population-level trajectories.

In [ ]:
FULL_NUC_RAW_PATHS: dict[str, dict[str, Path]] = {
    model: {
        landscape: RAW_DATA_DIR / f"figure6_full_nuc_gmu_{landscape}_{model}_raw.pkl"
        for landscape in LANDSCAPE_KEYS
    }
    for model in MODEL_KEYS
}


def get_missing_full_nuc_raw_paths(models: tuple[str, ...]) -> list[Path]:
    """Return missing full-nucleotide G_mu payload paths for a model group.

    Parameters:
    - models: tuple[str, ...]
        Mutation models to inspect.

    Returns:
    - list[Path]
        Missing raw-derived payload paths.
    """
    return [path for model in models for path in FULL_NUC_RAW_PATHS[model].values() if not path.exists()]


def generate_missing_full_nuc_raw_products(models: tuple[str, ...]) -> None:
    """Generate missing full-nucleotide G_mu payloads for the D-F landscapes.

    Parameters:
    - models: tuple[str, ...]
        Mutation models to pass to the raw generator.

    Returns:
    - None
        Writes raw-derived payloads when generation is requested.
    """
    command = [
        sys.executable,
        str(Path("scripts/generate_figure6_full_nuc_gmu.py")),
        "--landscapes",
        ",".join(LANDSCAPE_KEYS),
        "--models",
        ",".join(models),
        "--output-dir",
        str(RAW_DATA_DIR),
    ]
    if OVERWRITE_RAW_PKL:
        command.append("--overwrite")
    subprocess.run(command, check=True)


def report_full_nuc_raw_products(models: tuple[str, ...], label: str) -> list[Path]:
    """Print and return missing full-nucleotide products for a model group.

    Parameters:
    - models: tuple[str, ...]
        Mutation models required by a D-F panel group.
    - label: str
        Human-readable group label used in messages.

    Returns:
    - list[Path]
        Missing raw-derived payload paths after any requested generation.
    """
    missing_paths = get_missing_full_nuc_raw_paths(models)
    if not PLOT_ONLY and (missing_paths or OVERWRITE_RAW_PKL):
        generate_missing_full_nuc_raw_products(models)
        missing_paths = get_missing_full_nuc_raw_paths(models)
    print(f"Missing {label} full-nucleotide D-F raw products: {len(missing_paths)}")
    for path in missing_paths:
        print(f"  {path}")
    return missing_paths


def ensure_full_nuc_raw_products(models: tuple[str, ...], label: str) -> None:
    """Raise if raw full-nucleotide products are unavailable for processing.

    Parameters:
    - models: tuple[str, ...]
        Mutation models required by a D-F panel group.
    - label: str
        Human-readable label used in error messages.

    Returns:
    - None
        Raises when products are missing.
    """
    missing_paths = get_missing_full_nuc_raw_paths(models)
    if missing_paths:
        listing = "\n".join(str(path) for path in missing_paths)
        raise FileNotFoundError(f"Missing {label} full-nucleotide D-F raw products:\n{listing}")


missing_full_nuc_raw_paths = report_full_nuc_raw_products(MODEL_KEYS, "Figure S2 A. thaliana")


## Processing — Analytical Rates and Asymptotes for Panels D–F

Apply each product mutation kernel to the expanded nucleotide landscape. For the uniform and symmetric weighted kernels, calculate the squared-decay Rayleigh quotient around the unweighted mean. For the directed kernel, calculate the mean under the product stationary distribution and use the symmetrised forward/reverse operator. Store the analytical $\rho_2$, stationary squared mean $G_\infty$, and supporting terms for every empirical landscape.

In [ ]:
def build_nucleotide_landscape(landscape: np.ndarray) -> np.ndarray:
    """Expand an amino-acid landscape into nucleotide/codon space.

    Parameters:
    - landscape: np.ndarray
        Amino-acid landscape with one axis per residue.

    Returns:
    - np.ndarray
        Nucleotide landscape with three four-state axes per residue.
    """
    mapper = np.asarray(CODON_MAPPER, dtype=np.int16)
    buffered = np.pad(
        np.asarray(landscape, dtype=np.float64),
        [(0, 1)] * landscape.ndim,
        constant_values=float(np.min(landscape)),
    )
    indices = np.indices((4,) * (3 * landscape.ndim), dtype=np.int8)
    amino_acids = [
        mapper[indices[3 * site], indices[3 * site + 1], indices[3 * site + 2]]
        for site in range(landscape.ndim)
    ]
    return buffered[tuple(amino_acids)]


def apply_kernel_axis(values: np.ndarray, kernel: np.ndarray, axis: int) -> np.ndarray:
    """Apply one row-stochastic kernel to one function axis.

    Parameters:
    - values: np.ndarray
        Genotype-indexed function values.
    - kernel: np.ndarray
        Single-site row-stochastic kernel.
    - axis: int
        Axis receiving the kernel.

    Returns:
    - np.ndarray
        Kernel-transformed function values.
    """
    transformed = np.tensordot(kernel, values, axes=([1], [axis]))
    return np.moveaxis(transformed, 0, axis)


def stationary_average(values: np.ndarray, stationary: np.ndarray) -> float:
    """Compute the product-stationary average without forming a tensor product.

    Parameters:
    - values: np.ndarray
        Nucleotide-space landscape.
    - stationary: np.ndarray
        Single-site stationary distribution.

    Returns:
    - float
        Product-distribution weighted average.
    """
    contracted = np.asarray(values, dtype=np.float64)
    for _ in range(values.ndim):
        contracted = np.tensordot(stationary, contracted, axes=([0], [0]))
    return float(contracted)


def analytical_squared_decay(values: np.ndarray, kernel: np.ndarray, directed: bool) -> dict[str, object]:
    """Compute the analytical squared-decay metric for a product kernel.

    Parameters:
    - values: np.ndarray
        Nucleotide-space fitness landscape.
    - kernel: np.ndarray
        Single-site transition kernel.
    - directed: bool
        Whether to use the symmetrized directed Laplacian and stationary asymptote.

    Returns:
    - dict[str, object]
        Rate, asymptote, power term, and stationary distribution.
    """
    stationary = stationary_distribution(kernel)
    mean = stationary_average(values, stationary) if directed else float(values.mean())
    b_zero = float(values.size * mean * mean)
    denominator = float(np.vdot(values, values).real - b_zero)
    if denominator <= 0:
        raise ValueError("Squared-decay denominator must be positive.")
    laplacian_values = np.zeros_like(values, dtype=np.float64)
    for axis in range(values.ndim):
        forward = values - apply_kernel_axis(values, kernel, axis)
        if directed:
            reverse = values - apply_kernel_axis(values, kernel.T, axis)
            laplacian_values += 0.5 * (forward + reverse)
        else:
            laplacian_values += forward
    numerator = float(np.vdot(values, laplacian_values).real)
    rho_2 = numerator / (values.ndim * denominator)
    return {
        "rho_2": rho_2,
        "G_infinity": mean * mean,
        "b_0": b_zero,
        "stationary_distribution": stationary,
        "numerator_reduced": numerator,
    }


def analytics_complete(payload: dict[str, object], landscape_name: str) -> bool:
    """Return whether an analytics payload covers one landscape and all S2 models.

    Parameters:
    - payload: dict[str, object]
        Candidate analytics payload.
    - landscape_name: str
        Human-readable landscape name.

    Returns:
    - bool
        Whether every model entry is present for the landscape.
    """
    data = payload.get("data", {})
    return landscape_name in data and all(model in data[landscape_name] for model in MODEL_KEYS)


def build_landscape_analytics_payload(landscape_name: str) -> dict[str, object]:
    """Build analytical D-F rates for one landscape.

    Parameters:
    - landscape_name: str
        Human-readable landscape name.

    Returns:
    - dict[str, object]
        Per-landscape analytical D-F payload.
    """
    filename = LANDSCAPE_FILE_BY_NAME[landscape_name]
    with (REPO_ROOT / "landscape_arrays" / filename).open("rb") as handle:
        amino_acid_landscape = np.asarray(pickle.load(handle))
    nucleotide_landscape = build_nucleotide_landscape(amino_acid_landscape)
    return {
        "data": {
            landscape_name: {
                model: analytical_squared_decay(
                    nucleotide_landscape,
                    MUTATION_KERNELS[model],
                    directed=model.endswith("_directed"),
                )
                for model in MODEL_KEYS
            }
        },
        "params": {
            "landscape": landscape_name,
            "models": MODEL_KEYS,
            "kernels": MUTATION_KERNELS,
            "normalization": "sum(I-T_i) / N_nucleotide",
        },
        "metadata": {
            "paper_reference": "Figure S2D-F",
            "weighted_kernel": "symmetric Sinkhorn scaling of transpose-averaged nucleotide kernels",
            "directed_asymptote": "product stationary distribution of directed nucleotide kernels",
        },
    }


def load_or_build_landscape_analytics(landscape: str, landscape_name: str) -> dict[str, object]:
    """Load or build one per-landscape analytical D-F payload.

    Parameters:
    - landscape: str
        Short landscape key.
    - landscape_name: str
        Human-readable landscape name.

    Returns:
    - dict[str, object]
        Per-landscape analytical D-F payload.
    """
    path = ANALYTICS_PER_LANDSCAPE_PATHS[landscape]
    if path.exists():
        payload = load_pickle(path)
        if analytics_complete(payload, landscape_name):
            return payload
        if PLOT_ONLY:
            raise FileNotFoundError(f"PLOT_ONLY=True requires complete analytical payload: {path}")
    elif PLOT_ONLY:
        raise FileNotFoundError(f"PLOT_ONLY=True requires analytical payload: {path}")
    payload = build_landscape_analytics_payload(landscape_name)
    save_pickle(payload, path)
    return payload


landscape_analytics_payloads = {
    landscape: load_or_build_landscape_analytics(landscape, landscape_name)
    for landscape, landscape_name in zip(LANDSCAPE_KEYS, LANDSCAPE_NAMES, strict=True)
}
figure_s2_analytics = {
    "data": {
        landscape_name: landscape_analytics_payloads[landscape]["data"][landscape_name]
        for landscape, landscape_name in zip(LANDSCAPE_KEYS, LANDSCAPE_NAMES, strict=True)
    },
    "params": {
        "models": MODEL_KEYS,
        "kernels": MUTATION_KERNELS,
        "normalization": "sum(I-T_i) / N_nucleotide",
        "per_landscape_paths": {key: str(value) for key, value in ANALYTICS_PER_LANDSCAPE_PATHS.items()},
    },
    "metadata": {
        "paper_reference": "Figure S2D-F",
        "weighted_kernel": "symmetric Sinkhorn scaling of transpose-averaged nucleotide kernels",
        "directed_asymptote": "product stationary distribution of directed nucleotide kernels",
    },
}


## Processing — Start-Sampling Accuracy for Panels D–F

Normalise each compact $G_\mu$ curve by its initial value and fit the squared-decay model for every landscape, mutation kernel, start ordering, and prefix count. Convert the fitted exponent parameter to the reported $\rho_2$ and retain the 20 ordering-level estimates at each count for comparison with the analytical rate.

In [ ]:
def normalise_curve(curve: np.ndarray) -> np.ndarray:
    """Normalise a G_mu curve by its first generation value.

    Parameters:
    - curve: np.ndarray
        One-dimensional G_mu curve.

    Returns:
    - np.ndarray
        Normalised curve with finite values.
    """
    curve = np.asarray(curve, dtype=float)
    denominator = max(float(curve[0]), 1e-10)
    return curve / denominator


def fit_full_nuc_gmu_curve(curve: np.ndarray) -> float:
    """Fit one full-nucleotide G_mu curve and return rho_2.

    Parameters:
    - curve: np.ndarray
        Unnormalised G_mu curve.

    Returns:
    - float
        Fitted rho_2 value.
    """
    normalised = normalise_curve(curve)
    rho_raw = get_single_decay_rate_IK_v2(
        normalised,
        mut=DF_TOTAL_MUTATION_RATE,
        num_steps=DF_NUM_GENERATIONS,
    )[0]
    return float(rho_raw / 2.0)


def load_full_nuc_raw_by_model(landscape: str, models: tuple[str, ...]) -> dict[str, list[dict[str, object]]]:
    """Load compact full-nucleotide G_mu payloads for one landscape and model group.

    Parameters:
    - landscape: str
        Short landscape key.
    - models: tuple[str, ...]
        Mutation models to load.

    Returns:
    - dict[str, list[dict[str, object]]]
        Raw payloads keyed by model, each containing one landscape payload.
    """
    return {model: [load_pickle(FULL_NUC_RAW_PATHS[model][landscape])] for model in models}


def process_full_nuc_df_payload(
    raw_by_model: dict[str, list[dict[str, object]]],
    models: tuple[str, ...],
    landscape_keys: tuple[str, ...],
) -> dict[str, object]:
    """Fit full-nucleotide squared-decay rates for D-F style panels.

    Parameters:
    - raw_by_model: dict[str, list[dict[str, object]]]
        Compact full-nucleotide G_mu payloads keyed by mutation model.
    - models: tuple[str, ...]
        Models to process and preserve in output order.
    - landscape_keys: tuple[str, ...]
        Landscape keys corresponding to each payload.

    Returns:
    - dict[str, object]
        Per-model, per-landscape fitted-rate distributions and counts.
    """
    processed: dict[str, object] = {}
    counts_by_model: dict[str, list[np.ndarray]] = {}
    num_fit_failures = 0
    for model in models:
        model_results = []
        model_counts = []
        for landscape, payload in zip(landscape_keys, raw_by_model[model], strict=True):
            data = payload["data"]
            g_mu = np.asarray(data["g_mu"], dtype=float)
            counts = np.asarray(data["counts"], dtype=int)
            included_counts = np.asarray(data["included_counts"], dtype=int)
            if g_mu.shape != (20, len(counts), DF_NUM_GENERATIONS):
                raise ValueError(f"Unexpected g_mu shape for {landscape}/{model}: {g_mu.shape}")
            if not np.array_equal(included_counts, np.broadcast_to(counts[None, :], included_counts.shape)):
                raise ValueError(f"Included counts do not match counts for {landscape}/{model}.")
            if not np.isfinite(g_mu).all():
                raise ValueError(f"Non-finite g_mu values for {landscape}/{model}.")
            landscape_results = []
            for count_index in range(len(counts)):
                estimates = []
                for ordering_index in range(g_mu.shape[0]):
                    try:
                        estimates.append(fit_full_nuc_gmu_curve(g_mu[ordering_index, count_index]))
                    except RuntimeError:
                        num_fit_failures += 1
                        estimates.append(np.nan)
                landscape_results.append(np.asarray(estimates, dtype=float))
            model_results.append(landscape_results)
            model_counts.append(counts)
        processed[model] = model_results
        counts_by_model[model] = model_counts
    return {
        "data": processed,
        "counts": counts_by_model,
        "params": {
            "landscapes": landscape_keys,
            "models": models,
            "M": DF_NUM_GENERATIONS,
            "total_mutation_rate": DF_TOTAL_MUTATION_RATE,
            "num_orderings": 20,
            "kernels": {model: MUTATION_KERNELS[model] for model in models},
            "raw_paths": {
                model: [str(FULL_NUC_RAW_PATHS[model][landscape]) for landscape in landscape_keys]
                for model in models
            },
        },
        "metadata": {
            "paper_reference": "Figure S2D-F",
            "description": "Full-nucleotide compact G_mu start-prefix fitted rho_2 distributions.",
            "num_fit_failures": num_fit_failures,
            "standard_deviation_ddof": 1,
        },
    }


def load_or_process_landscape_df_payload(landscape: str, path: Path) -> dict[str, object]:
    """Load or process one per-landscape full-nucleotide D-F payload.

    Parameters:
    - landscape: str
        Short landscape key.
    - path: Path
        Per-landscape processed payload destination.

    Returns:
    - dict[str, object]
        Processed D-F payload containing one landscape.
    """
    if path.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
        return load_pickle(path)
    if PLOT_ONLY:
        raise FileNotFoundError(f"PLOT_ONLY=True requires {path}")
    ensure_full_nuc_raw_products(MODEL_KEYS, "Figure S2 A. thaliana")
    payload = process_full_nuc_df_payload(load_full_nuc_raw_by_model(landscape, MODEL_KEYS), MODEL_KEYS, (landscape,))
    save_pickle(payload, path)
    return payload


def combine_landscape_df_payloads(payloads: dict[str, dict[str, object]]) -> dict[str, object]:
    """Combine per-landscape D-F payloads in memory for plotting.

    Parameters:
    - payloads: dict[str, dict[str, object]]
        Per-landscape payloads keyed by short landscape key.

    Returns:
    - dict[str, object]
        Combined plotting payload.
    """
    data: dict[str, list[object]] = {model: [] for model in MODEL_KEYS}
    counts: dict[str, list[np.ndarray]] = {model: [] for model in MODEL_KEYS}
    raw_paths: dict[str, list[str]] = {model: [] for model in MODEL_KEYS}
    num_fit_failures = 0
    for landscape in LANDSCAPE_KEYS:
        payload = payloads[landscape]
        for model in MODEL_KEYS:
            data[model].append(payload["data"][model][0])
            counts[model].append(np.asarray(payload["counts"][model][0], dtype=int))
            raw_paths[model].extend(payload.get("params", {}).get("raw_paths", {}).get(model, []))
        num_fit_failures += int(payload.get("metadata", {}).get("num_fit_failures", 0))
    return {
        "data": data,
        "counts": counts,
        "params": {
            "models": MODEL_KEYS,
            "M": DF_NUM_GENERATIONS,
            "total_mutation_rate": DF_TOTAL_MUTATION_RATE,
            "num_orderings": 20,
            "kernels": {model: MUTATION_KERNELS[model] for model in MODEL_KEYS},
            "raw_paths": raw_paths,
            "per_landscape_paths": {key: str(value) for key, value in DF_PER_LANDSCAPE_PROCESSED_PATHS.items()},
        },
        "metadata": {
            "paper_reference": "Figure S2D-F",
            "description": "Combined in-memory view of per-landscape full-nucleotide fitted rho_2 distributions.",
            "num_fit_failures": num_fit_failures,
            "standard_deviation_ddof": 1,
        },
    }


landscape_df_payloads = {
    landscape: load_or_process_landscape_df_payload(
        landscape,
        DF_PER_LANDSCAPE_PROCESSED_PATHS[landscape],
    )
    for landscape in LANDSCAPE_KEYS
}
figure_s2_df_payload = combine_landscape_df_payloads(landscape_df_payloads)


## Plotting Functions

Define the renderers for the synthetic NK local-decay summaries and the empirical start-sampling analyses. The functions use the processed means, standard deviations, and analytical references without recomputing decay curves.

In [ ]:
def local_symbol(model: str) -> str:
    """Return the plotted local rho_2 symbol for an A-C mutation model.

    Parameters:
    - model: str
        Mutation-model key.

    Returns:
    - str
        Matplotlib mathtext label.
    """
    if model == "nuc_uniform":
        return r"$\rho_2^{\mathrm{loc}}$"
    if model.endswith("_weighted"):
        return r"$\widetilde{\rho}_2^{\mathrm{loc}}$"
    return r"$\overline{\rho}_2^{\mathrm{loc}}$"


def df_symbol(model: str, *, fitted: bool) -> str:
    """Return the plotted rho_2 symbol for a D-F mutation model.

    Parameters:
    - model: str
        Mutation-model key.
    - fitted: bool
        Whether the fitted-rate superscript should be included.

    Returns:
    - str
        Matplotlib mathtext label.
    """
    suffix = r"^{\mathrm{fit}}" if fitted else ""
    if model == "nuc_uniform":
        return rf"$\rho_2{suffix}$"
    if model.endswith("_weighted"):
        return rf"$\widetilde{{\rho}}_2{suffix}$"
    return rf"$\overline{{\rho}}_2{suffix}$"


def plot_abc_panel(ax: Axes, payload: dict[str, object], model: str) -> None:
    """Plot one synthetic-NK local squared-decay panel.

    Parameters:
    - ax: Axes
        Axis receiving the plot.
    - payload: dict[str, object]
        Processed A-C payload.
    - model: str
        Mutation-model key.

    Returns:
    - None
        Artists are added to the axis.
    """
    panel = payload["data"][model]
    rho = np.asarray(panel["binned_rho_NK"])
    means = np.asarray(panel["binned_mean"])
    stds = np.asarray(panel["binned_std"])
    symbol = local_symbol(model)
    ax.plot(rho, means, "o-", linewidth=1.4, markersize=4, label=symbol)
    ax.fill_between(rho, means - stds, means + stds, alpha=0.25, linewidth=0)
    ax.plot((0, 1), (0, 1), color="red", linestyle="--", alpha=0.55, label=r"$\rho_{NK}$")
    ax.set_title(MODEL_TITLES[model], fontsize=FIGURE_TITLE_SIZE)
    ax.set_xlabel(r"$\rho_{NK}$", fontsize=FIGURE_LABEL_SIZE)
    ax.set_ylabel(symbol, fontsize=FIGURE_LABEL_SIZE)
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.10)
    ax.legend(fontsize=FIGURE_LEGEND_SIZE, loc="upper left")
    ax.tick_params(labelsize=FIGURE_TICK_SIZE)
    ax.grid(True, alpha=0.18)


def plot_df_panel(ax: Axes, payload: dict[str, object], model: str) -> None:
    """Plot sampling accuracy and its analytical reference for one kernel.

    Parameters:
    - ax: Axes
        Axis receiving the plot.
    - payload: dict[str, object]
        Processed D-F full-nucleotide payload.
    - model: str
        Mutation-model key.

    Returns:
    - None
        Artists are added to the axis.
    """
    symbol = df_symbol(model, fitted=True)
    reference_symbol = df_symbol(model, fitted=False)
    max_start_count = 1
    for index, (name, color, marker) in enumerate(zip(
        LANDSCAPE_NAMES, LANDSCAPE_COLORS, LANDSCAPE_MARKERS, strict=True
    )):
        starts = np.asarray(payload["counts"][model][index], dtype=float)
        values = payload["data"][model][index]
        means = np.asarray([np.nanmean(item) for item in values])
        stds = np.asarray([np.nanstd(item, ddof=1) for item in values])
        max_start_count = max(max_start_count, int(np.nanmax(starts)))
        ax.plot(starts, means, color=color, marker=marker, markersize=3.5,
                markevery=2, linewidth=1.3, label=name)
        ax.fill_between(starts, means - stds, means + stds, color=color, alpha=0.15)
        reference = float(figure_s2_analytics["data"][name][model]["rho_2"])
        ax.axhline(reference, color=color, linestyle="--", linewidth=1.0, alpha=0.65)
    ax.set_xscale("log")
    ax.set_title(MODEL_TITLES[model], fontsize=FIGURE_TITLE_SIZE)
    ax.set_xlabel("Number of starting points", fontsize=FIGURE_LABEL_SIZE)
    ax.set_ylabel(symbol, fontsize=FIGURE_LABEL_SIZE)
    ax.set_xlim(1, max_start_count)
    ax.set_ylim(0, 1.25)
    handles, labels = ax.get_legend_handles_labels()
    handles.append(mlines.Line2D([], [], color="black", linestyle="--", label=reference_symbol))
    labels.append(reference_symbol)
    ax.legend(handles, labels, fontsize=FIGURE_LEGEND_SIZE, loc="upper right")
    ax.tick_params(labelsize=FIGURE_TICK_SIZE)


## Individual Panels A–C

Plot binned local squared-decay estimates against $\rho_{NK}$ for the uniform, symmetric weighted, and directed A. thaliana mutation kernels.

In [ ]:
for letter, model in zip("ABC", MODEL_KEYS, strict=True):
    fig, ax = plt.subplots(figsize=(3.2, 2.8), dpi=PANEL_DPI)
    plot_abc_panel(ax, figure_s2_abc_payload, model)
    add_panel_letter(ax, letter)
    if SAVE_FIGURES:
        save_figure(fig, f"figure_S2{letter}")
    plt.show()


## Individual Panels D–F

Plot the fitted empirical squared-decay rates against the number of starting genotypes for each mutation model, with separate landscapes and analytical horizontal references.

In [ ]:
for letter, model in zip("DEF", MODEL_KEYS, strict=True):
    fig, ax = plt.subplots(figsize=(3.2, 2.8), dpi=PANEL_DPI)
    plot_df_panel(ax, figure_s2_df_payload, model)
    add_panel_letter(ax, letter)
    if SAVE_FIGURES:
        save_figure(fig, f"figure_S2{letter}")
    plt.show()


## Complete Figure

Arrange the three synthetic NK panels and three full-nucleotide empirical sampling panels in the complete Figure S2 layout.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9.6, 5.8), dpi=PANEL_DPI, constrained_layout=True)
for ax, letter, model in zip(axes[0], "ABC", MODEL_KEYS, strict=True):
    plot_abc_panel(ax, figure_s2_abc_payload, model)
    add_panel_letter(ax, letter)
for ax, letter, model in zip(axes[1], "DEF", MODEL_KEYS, strict=True):
    plot_df_panel(ax, figure_s2_df_payload, model)
    add_panel_letter(ax, letter)
if SAVE_FIGURES:
    save_figure(fig, "figure_S2")
plt.show()
